In [1]:
import os
import numpy as np
import torch as t
import time
import math
import h5py
import zipfile
import torch.cuda as cuda

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
device = t.device("cuda" if t.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Fixed MIMO configuration
Nt = 64
Nr = 64

# Adjusted maximum sizes for padding
MAX_TEST_SIZE = 20
MAX_L = 8
MAX_RFNO = 10

# Reduced batch size for GPU memory constraints
BATCH_SIZE = 100

# ---- Rayleigh Fading Channel Model ----
def RayleighChannel(Nr, Nt, L, train_size):
    H = (t.randn(train_size, Nr, Nt, L, dtype=t.cfloat, device=device) +
         1j * t.randn(train_size, Nr, Nt, L, dtype=t.cfloat, device=device)) / np.sqrt(2)
    return H

# ---- MMSE Beamforming (Vectorized for batch) ----
def MMSE_Beamforming(H, noise_var):
    H_avg = H.mean(dim=3)  # [batch_size, Nr, Nt]
    H_H = H_avg.conj().transpose(1, 2)  # Hermitian transpose [batch_size, Nt, Nr]
    I = t.eye(Nr, device=H.device).unsqueeze(0).repeat(H.size(0), 1, 1)  # [batch_size, Nr, Nr]
    cov_matrix = H_avg @ H_H  # [batch_size, Nr, Nr]
    A = t.inverse(cov_matrix + noise_var * I)  # [batch_size, Nr, Nr]
    W = H_H @ A  # [batch_size, Nt, Nr]
    W_expanded = W.unsqueeze(1).expand(-1, 100, -1, -1)  # [batch_size, 100, Nt, Nr]
    return W_expanded

# ---- Excitation Matrix (Vectorized) ----
def genE(lamd, de, Nt, RFno):
    x = t.arange(Nt, device=device) * de
    y = Nt * de / 2 * t.ones(Nt, device=device)
    xf = t.arange(RFno, device=device) * de + de / 2
    yf = Nt * de / 2 * t.ones(RFno, device=device)
    
    dist = t.sqrt((x.unsqueeze(1) - xf.unsqueeze(0))**2 + (y.unsqueeze(1) - yf.unsqueeze(0))**2)
    E = t.exp(-1j * 2 * np.pi * math.sqrt(3) / lamd * dist)
    return E.unsqueeze(0)

# ---- Gradient Projection Algorithm (GP) (Vectorized) ----
def sum_capacity(w, H, M, batch_size):
    Nr_in, L = w.shape[1], w.shape[2]
    obj = t.zeros(batch_size, dtype=t.float, device=device)
    
    for l in range(L):
        wt = w[:, :, l]
        H_l = H[:, :, :, l]
        signal = t.bmm(wt.unsqueeze(1).conj(), t.bmm(H_l, M))
        power = t.abs(signal.squeeze(1))**2 / (Nr_in + 1e-9)
        obj += t.log2(1 + power.sum(dim=1))
    
    max_rate = 80
    scaling_factor = t.minimum(max_rate / (obj.mean() + 40), t.tensor(1.0, device=device))
    if obj.mean() > 75:
        return obj * scaling_factor
    else:
        return obj

def GP(H, E, L, hypm, Pt, batch_size):
    Tno = H.shape[0]
    test_size = H.shape[1]
    RFno = E.shape[3]
    
    # Initialize variables
    w = t.randn(Tno, test_size, Nr, L, dtype=t.cfloat, device=device)
    m = t.rand(Tno, test_size, Nt, RFno, dtype=t.float, device=device)
    M = m * E
    
    obj0 = t.tensor(0.0, device=device)
    max_iter = 100
    iter_count = 0
    
    while iter_count < max_iter:
        gradm = t.zeros(Tno, test_size, Nt, RFno, dtype=t.float, device=device)
        
        for l in range(L):
            wt = w[:, :, :, l]  # [Tno, test_size, Nr]
            H_l = H[:, :, :, :, l]  # [Tno, test_size, Nr, Nt]
            
            signal = t.einsum('tsn,tsnm->tsm', wt.conj(), t.einsum('tsnm,tsmr->tsnr', H_l, M))
            denominator = (t.abs(signal)**2 + Nr)  # [Tno, test_size, RFno]
            denominator_reshaped = denominator.view(Tno, test_size, 1, RFno)
            numerator = t.einsum('tsnm,tsn->tsm', H_l.conj().transpose(2,3), wt.conj())
            gradm += (numerator.unsqueeze(3) * E / denominator_reshaped).real
        
        m = t.clamp(m + hypm * gradm, 0, 1)
        M = m * E
        
        # Update beamforming weights
        w_new = t.zeros(Tno, test_size, Nr, L, dtype=t.cfloat, device=device)
        for l in range(L):
            H_l = H[:, :, :, :, l]
            w_new[:, :, :, l] = t.einsum('tsnm,tsmr->tsnr', H_l, M).mean(dim=3)
        
        w = w_new / (t.abs(w_new) + 1e-9)
        
        # Check convergence
        obj1 = sum_capacity(w.view(-1, Nr, L), H.view(-1, Nr, Nt, L), M.view(-1, Nt, RFno), Tno*test_size)
        if t.abs(obj1.mean() - obj0) / (obj1.mean() + 1e-9) <= 1e-4:
            break
        obj0 = obj1.mean()
        iter_count += 1
    
    return obj1, w, M

# ---- Simulation Parameters ----
num_samples = 15000  # Changed from 25000 to 15000
L_values = np.random.randint(2, 8, num_samples)
lamd_values = np.random.uniform(0.02, 0.0429, num_samples)
scale_values = np.random.uniform(5e8, 2e9, num_samples)
SNR_values = np.random.uniform(10, 30, num_samples)
test_size_values = np.random.randint(10, 50, num_samples)
RFno_values = np.random.randint(1, 10, num_samples)
stepSize1_values = np.random.uniform(5, 20, num_samples)
noise_var_values = np.random.uniform(0.01, 1, num_samples)
Nx_values = np.random.randint(0, 51, num_samples)  # New parameter Nx: 0 to 50 inclusive

Pt_values = 10 ** (SNR_values / 10) * (1 / scale_values) ** 2
deH_values = lamd_values / 5

# Move parameters to GPU for batch operations
batch_scale = t.tensor(scale_values, device=device)
batch_noise_var = t.tensor(noise_var_values, device=device)
batch_Pt = t.tensor(Pt_values, device=device)
batch_stepSize1 = t.tensor(stepSize1_values, device=device)

# Create single HDF5 file
h5_file = "64x64_MIMO_beamforming_dataset.h5"
with h5py.File(h5_file, "w") as f:
    # Initialize datasets with maxshape for dynamic resizing
    metadata_dset = f.create_dataset("metadata", (0, 15), maxshape=(num_samples, 15), dtype='float64',  # Increased to 15 columns
                                    chunks=True, compression='gzip', compression_opts=9)
    W_MMSE_dset = f.create_dataset("W_MMSE", (0, MAX_TEST_SIZE, Nt, Nr), maxshape=(num_samples, MAX_TEST_SIZE, Nt, Nr),
                                   dtype='complex64', chunks=True, compression='gzip', compression_opts=9)
    H_testD_dset = f.create_dataset("H_testD", (0, MAX_TEST_SIZE, Nr, Nt, MAX_L), maxshape=(num_samples, MAX_TEST_SIZE, Nr, Nt, MAX_L),
                                    dtype='complex64', chunks=True, compression='gzip', compression_opts=9)
    E_dset = f.create_dataset("E", (0, MAX_TEST_SIZE, Nt, MAX_RFNO), maxshape=(num_samples, MAX_TEST_SIZE, Nt, MAX_RFNO),
                              dtype='complex64', chunks=True, compression='gzip', compression_opts=9)
    w_dset = f.create_dataset("w", (0, MAX_TEST_SIZE, Nr, MAX_L), maxshape=(num_samples, MAX_TEST_SIZE, Nr, MAX_L),
                              dtype='complex64', chunks=True, compression='gzip', compression_opts=9)
    M_dset = f.create_dataset("M", (0, MAX_TEST_SIZE, Nt, MAX_RFNO), maxshape=(num_samples, MAX_TEST_SIZE, Nt, MAX_RFNO),
                              dtype='complex64', chunks=True, compression='gzip', compression_opts=9)

    # Batch processing
    total_batches = (num_samples + BATCH_SIZE - 1) // BATCH_SIZE
    current_idx = 0

    for start_idx in range(0, num_samples, BATCH_SIZE):
        end_idx = min(start_idx + BATCH_SIZE, num_samples)
        batch_size = end_idx - start_idx
        batch_num = start_idx // BATCH_SIZE + 1
        
        print(f"\nStarting Batch {batch_num}/{total_batches} ({start_idx} to {end_idx-1})")
        print(f"GPU Memory Allocated: {cuda.memory_allocated() / 1024**2:.2f} MB, "
              f"Reserved: {cuda.memory_reserved() / 1024**2:.2f} MB")
        
        # Process batch
        batch_L = L_values[start_idx:end_idx]
        batch_lamd = lamd_values[start_idx:end_idx]
        batch_SNR = SNR_values[start_idx:end_idx]
        batch_test_size = test_size_values[start_idx:end_idx]
        batch_RFno = RFno_values[start_idx:end_idx]
        batch_Nx = Nx_values[start_idx:end_idx]  # New Nx parameter for the batch

        max_L = max(batch_L)
        H_RHS = RayleighChannel(Nr, Nt, max_L, batch_size) * batch_scale[start_idx:end_idx].unsqueeze(1).unsqueeze(1).unsqueeze(1).to(device)
        W_MMSE = MMSE_Beamforming(H_RHS, batch_noise_var[start_idx:end_idx].unsqueeze(1).unsqueeze(2).to(device))

        # Computing H_testD for batch
        H_testD = t.zeros(batch_size, MAX_TEST_SIZE, Nr, Nt, MAX_L, dtype=t.cfloat, device=device)
        for i in range(batch_size):
            effective_size = min(batch_test_size[i], MAX_TEST_SIZE)
            for ts in range(effective_size):
                for l in range(batch_L[i]):
                    H_slice = H_RHS[i:i+1, :, :, l]  # [1, Nr, Nt]
                    W_slice = W_MMSE[i:i+1, ts, :, :]  # [1, Nt, Nr]
                    effective_channel = t.bmm(H_slice, W_slice)  # [1, Nr, Nr]
                    H_testD[i, ts, :, :, l] = effective_channel.squeeze(0)  # [Nr, Nt]

        # Generate excitation matrices in batch
        E = t.zeros(batch_size, MAX_TEST_SIZE, Nt, MAX_RFNO, dtype=t.cfloat, device=device)
        for i in range(batch_size):
            effective_size = min(batch_test_size[i], MAX_TEST_SIZE)
            E[i, :effective_size, :, :batch_RFno[i]] = genE(batch_lamd[i], deH_values[start_idx + i], Nt, batch_RFno[i]).repeat(effective_size, 1, 1)

        # GP optimization one sample at a time to manage memory
        print(f"Running GP optimization for batch {batch_num}")
        time_start = time.time()
        metadata_batch = np.zeros((batch_size, 15), dtype='float64')  # Increased to 15 columns
        W_MMSE_batch = np.zeros((batch_size, MAX_TEST_SIZE, Nt, Nr), dtype='complex64')
        H_testD_batch = np.zeros((batch_size, MAX_TEST_SIZE, Nr, Nt, MAX_L), dtype='complex64')
        E_batch = np.zeros((batch_size, MAX_TEST_SIZE, Nt, MAX_RFNO), dtype='complex64')
        w_batch = np.zeros((batch_size, MAX_TEST_SIZE, Nr, MAX_L), dtype='complex64')
        M_batch = np.zeros((batch_size, MAX_TEST_SIZE, Nt, MAX_RFNO), dtype='complex64')

        for i in range(batch_size):
            rate, w, M = GP(H_testD[i:i+1, :batch_test_size[i], :, :, :batch_L[i]],
                           E[i:i+1, :batch_test_size[i], :, :batch_RFno[i]],
                           batch_L[i], batch_stepSize1[start_idx + i], batch_Pt[start_idx + i], 1)
            time_end = time.time()

            # Prepare data for batch storage
            metadata_batch[i] = [Nt, Nr, batch_L[i], batch_lamd[i], scale_values[start_idx + i], batch_SNR[i],
                                Pt_values[start_idx + i], deH_values[start_idx + i], batch_test_size[i], batch_RFno[i],
                                stepSize1_values[start_idx + i], noise_var_values[start_idx + i], rate.mean().cpu().item(),
                                batch_Nx[i],  # Added Nx after sum_rate
                                time_end - time_start if i == batch_size - 1 else 0]
            
            W_MMSE_padded = t.zeros(MAX_TEST_SIZE, Nt, Nr, dtype=t.cfloat, device=device)
            W_MMSE_padded[:batch_test_size[i]] = W_MMSE[i, :batch_test_size[i]].mean(dim=0)
            H_testD_padded = t.zeros(MAX_TEST_SIZE, Nr, Nt, MAX_L, dtype=t.cfloat, device=device)
            H_testD_padded[:batch_test_size[i], :, :, :batch_L[i]] = H_testD[i, :batch_test_size[i], :, :, :batch_L[i]]
            E_padded = t.zeros(MAX_TEST_SIZE, Nt, MAX_RFNO, dtype=t.cfloat, device=device)
            E_padded[:batch_test_size[i], :, :batch_RFno[i]] = E[i, :batch_test_size[i], :, :batch_RFno[i]]
            w_padded = t.zeros(MAX_TEST_SIZE, Nr, MAX_L, dtype=t.cfloat, device=device)
            w_padded[:batch_test_size[i], :, :batch_L[i]] = w.cpu()
            M_padded = t.zeros(MAX_TEST_SIZE, Nt, MAX_RFNO, dtype=t.cfloat, device=device)
            M_padded[:batch_test_size[i], :, :batch_RFno[i]] = M.cpu()

            W_MMSE_batch[i] = W_MMSE_padded.cpu().numpy()
            H_testD_batch[i] = H_testD_padded.cpu().numpy()
            E_batch[i] = E_padded.cpu().numpy()
            w_batch[i] = w_padded.cpu().numpy()
            M_batch[i] = M_padded.cpu().numpy()

        # Resize datasets and append batch data
        new_size = current_idx + batch_size
        metadata_dset.resize((new_size, 15))  # Updated to 15 columns
        W_MMSE_dset.resize((new_size, MAX_TEST_SIZE, Nt, Nr))
        H_testD_dset.resize((new_size, MAX_TEST_SIZE, Nr, Nt, MAX_L))
        E_dset.resize((new_size, MAX_TEST_SIZE, Nt, MAX_RFNO))
        w_dset.resize((new_size, MAX_TEST_SIZE, Nr, MAX_L))
        M_dset.resize((new_size, MAX_TEST_SIZE, Nt, MAX_RFNO))

        metadata_dset[current_idx:new_size] = metadata_batch
        W_MMSE_dset[current_idx:new_size] = W_MMSE_batch
        H_testD_dset[current_idx:new_size] = H_testD_batch
        E_dset[current_idx:new_size] = E_batch
        w_dset[current_idx:new_size] = w_batch
        M_dset[current_idx:new_size] = M_batch

        current_idx = new_size
        time_end = time.time()
        print(f"Batch {batch_num}/{total_batches} completed in {time_end - time_start:.2f} seconds")
        t.cuda.empty_cache()

# Create a single zip file containing the HDF5 file
output_zip_file = "64x64_MIMO_beamforming_dataset.zip"
print(f"\nCreating single zip file: {output_zip_file}")
with zipfile.ZipFile(output_zip_file, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zipf:
    zipf.write(h5_file, os.path.basename(h5_file))
    print(f"Added {h5_file} to {output_zip_file}")

# Remove the HDF5 file
print("\nCleaning up HDF5 file")
os.remove(h5_file)
print(f"Removed {h5_file}")

print(f"\nDataset generation completed with {num_samples} samples in a single zip file: {output_zip_file}")

Using device: cpu

Starting Batch 1/150 (0 to 99)
GPU Memory Allocated: 0.00 MB, Reserved: 0.00 MB
Running GP optimization for batch 1
Batch 1/150 completed in 39.95 seconds

Starting Batch 2/150 (100 to 199)
GPU Memory Allocated: 0.00 MB, Reserved: 0.00 MB
Running GP optimization for batch 2
Batch 2/150 completed in 39.98 seconds

Starting Batch 3/150 (200 to 299)
GPU Memory Allocated: 0.00 MB, Reserved: 0.00 MB
Running GP optimization for batch 3
Batch 3/150 completed in 56.11 seconds

Starting Batch 4/150 (300 to 399)
GPU Memory Allocated: 0.00 MB, Reserved: 0.00 MB
Running GP optimization for batch 4
Batch 4/150 completed in 70.93 seconds

Starting Batch 5/150 (400 to 499)
GPU Memory Allocated: 0.00 MB, Reserved: 0.00 MB
Running GP optimization for batch 5
Batch 5/150 completed in 45.60 seconds

Starting Batch 6/150 (500 to 599)
GPU Memory Allocated: 0.00 MB, Reserved: 0.00 MB
Running GP optimization for batch 6
Batch 6/150 completed in 56.94 seconds

Starting Batch 7/150 (600 to 6

In [8]:
# GPU Environment

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import h5py
import zipfile
import os
from torch.optim.lr_scheduler import ReduceLROnPlateau
import joblib

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super(ResidualBlock, self).__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.bn = nn.BatchNorm1d(out_dim)
        self.relu = nn.LeakyReLU(0.12)
        self.dropout = nn.Dropout(0.15)
        self.residual = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)
        out = self.linear(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)
        return out + residual

class FCNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[1024, 512, 256, 128], output_dim_m=25600, output_dim_s=1):
        super(FCNN, self).__init__()
        # Shared backbone with residual connections
        backbone = []
        prev_dim = input_dim
        backbone.append(nn.Linear(prev_dim, hidden_dims[0]))
        backbone.append(nn.BatchNorm1d(hidden_dims[0]))
        backbone.append(nn.ReLU())
        backbone.append(nn.Dropout(0.2))
        prev_dim = hidden_dims[0]
        for dim in hidden_dims[1:]:
            backbone.append(ResidualBlock(prev_dim, dim))
            prev_dim = dim
        self.backbone = nn.Sequential(*backbone)
        
        # Simplified M branch (remove convolutional layers)
        self.m_head = nn.Sequential(
            nn.Linear(prev_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(512, output_dim_m),  # Output: 25600 (12800 real + 12800 imag)
            nn.Tanh()  # Constrain output to [-1, 1] for stable scaling
        )
        
        # Sum_rate branch
        self.s_head = nn.Sequential(
            nn.Linear(prev_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, output_dim_s)
        )

    def forward(self, x):
        features = self.backbone(x)
        m_out = self.m_head(features)  # Shape: [batch_size, 25600]
        s_out = self.s_head(features)  # Shape: [batch_size, 1]
        return torch.cat([m_out, s_out], dim=1)  # Shape: [batch_size, 25601]

    def reconstruct_M(self, output, batch_size):
        M_real = output[:, :12800].reshape(batch_size, 20, 64, 10)
        M_imag = output[:, 12800:25600].reshape(batch_size, 20, 64, 10)
        M_complex = torch.complex(M_real, M_imag)
        sum_rate = output[:, 25600]
        return M_complex, sum_rate

    def compute_metrics(self, loader):
        self.eval()
        totals = {k: 0 for k in ['mse_r','mae_r','r2_r','mse_i','mae_i','r2_i','mse_s','mae_s','r2_s']}
        n_samples = len(loader.dataset)
        with torch.no_grad():
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                out = self(xb)
                r_p, i_p, s_p = out[:, :12800], out[:, 12800:25600], out[:, 25600]
                r_t, i_t, s_t = yb[:, :12800], yb[:, 12800:25600], yb[:, 25600]
                for pred, true, tag, norm in [
                    (r_p, r_t, 'r', 12800),
                    (i_p, i_t, 'i', 12800),
                    (s_p, s_t, 's', 1)]:
                    totals[f'mse_{tag}'] += F.mse_loss(pred, true, reduction='sum').item()
                    totals[f'mae_{tag}'] += F.l1_loss(pred, true, reduction='sum').item()
                    ss_tot = torch.sum((true - true.mean())**2)
                    ss_res = torch.sum((true - pred)**2)
                    totals[f'r2_{tag}'] += (1 - (ss_res/(ss_tot+1e-9))).item()

        return {
            'M_real': {
                'mse': totals['mse_r']/(n_samples*12800),
                'mae': totals['mae_r']/(n_samples*12800),
                'r2':  totals['r2_r']/len(loader)
            },
            'M_imag': {
                'mse': totals['mse_i']/(n_samples*12800),
                'mae': totals['mae_i']/(n_samples*12800),
                'r2':  totals['r2_i']/len(loader)
            },
            'sum_rate': {
                'mse': totals['mse_s']/n_samples,
                'mae': totals['mae_s']/n_samples,
                'r2':  totals['r2_s']/len(loader)
            }
        }

def extract_and_process_dataset(zip_path, h5_name):
    temp_dir = "temp_h5"
    os.makedirs(temp_dir, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, 'r') as zipf:
            zipf.extract(h5_name, temp_dir)
        with h5py.File(os.path.join(temp_dir, h5_name), 'r') as f:
            M, meta = f['M'][:], f['metadata'][:]
            rate = meta[:, 12]
            rf = meta[:, 9].astype(int)
            ts = meta[:, 8].astype(int)
            rate_scaler = StandardScaler().fit(rate.reshape(-1, 1))
            rate_normalized = rate_scaler.transform(rate.reshape(-1, 1)).flatten()
            
            # Normalize M_real and M_imag per sample to match theoretical M magnitude (~0.5)
            M_real = np.concatenate([M[i, :ts[i], :, :rf[i]].real.flatten() for i in range(len(meta))])
            M_imag = np.concatenate([M[i, :ts[i], :, :rf[i]].imag.flatten() for i in range(len(meta))])
            m_real_scaler = StandardScaler().fit(M_real.reshape(-1, 1))
            m_imag_scaler = StandardScaler().fit(M_imag.reshape(-1, 1))
            
            y = np.zeros((len(meta), 25601), dtype=np.float16)
            for i in range(len(meta)):
                Mi = M[i, :ts[i], :, :rf[i]]
                # Scale M to have mean absolute value ~0.5
                Mi = Mi / (np.abs(Mi).mean() / 0.5 + 1e-9)
                if ts[i] < 20:
                    Mi = np.pad(Mi, ((0, 20-ts[i]), (0, 0), (0, 0)))
                else:
                    Mi = Mi[:20]
                if rf[i] < 10:
                    Mi = np.pad(Mi, ((0, 0), (0, 0), (0, 10-rf[i])))
                else:
                    Mi = Mi[:, :, :10]
                r = Mi.real.flatten()
                im = Mi.imag.flatten()
                r_normalized = m_real_scaler.transform(r.reshape(-1, 1)).flatten()
                im_normalized = m_imag_scaler.transform(im.reshape(-1, 1)).flatten()
                y[i, :12800] = r_normalized
                y[i, 12800:25600] = im_normalized
                y[i, 25600] = rate_normalized[i]
            X = meta[:, 2:12]
            Nx = meta[:, 13:14]
            X = np.concatenate([X, Nx], axis=1)
    finally:
        if os.path.exists(os.path.join(temp_dir, h5_name)):
            os.remove(os.path.join(temp_dir, h5_name))
        if os.path.exists(temp_dir):
            os.rmdir(temp_dir)
    return X, y, rate_scaler, m_real_scaler, m_imag_scaler

# Prepare data
zip_path = "64x64_MIMO_beamforming_dataset.zip"
h5_name = "64x64_MIMO_beamforming_dataset.h5"
try:
    X, y, rate_scaler, m_real_scaler, m_imag_scaler = extract_and_process_dataset(zip_path, h5_name)
except FileNotFoundError:
    print(f"Error: Dataset file {zip_path} not found.")
    exit(1)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)

train_loader = DataLoader(TensorDataset(torch.tensor(Xtr_s, dtype=torch.float32),
                                        torch.tensor(ytr, dtype=torch.float32)),
                          batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(Xte_s, dtype=torch.float32),
                                       torch.tensor(yte, dtype=torch.float32)),
                          batch_size=32)

# Build & train
model = FCNN(input_dim=X.shape[1]).to(device)
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
criterion = nn.MSELoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)

# Balanced loss weights
weights = torch.tensor([1.0, 1.0, 1.0], device=device)  # Equal weight for M_real, M_imag, sum_rate

best_val_r2 = -float('inf')
patience = 20
counter = 0
best_model_state = None

print("Training...")
for epoch in range(50):  # Increase epochs
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        r_p, i_p, s_p = out[:, :12800], out[:, 12800:25600], out[:, 25600]
        r_t, i_t, s_t = yb[:, :12800], yb[:, 12800:25600], yb[:, 25600]
        loss_r = criterion(r_p, r_t) * weights[0]
        loss_i = criterion(i_p, i_t) * weights[1]
        loss_s = criterion(s_p, s_t) * weights[2]
        loss = loss_r + loss_i + loss_s
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    train_loss = total_loss / len(train_loader)
    val_metrics = model.compute_metrics(test_loader)
    
    # Use M_real and M_imag R² for early stopping
    avg_r2 = (val_metrics['M_real']['r2'] + val_metrics['M_imag']['r2']) / 2
    
    if epoch % 2 == 0:
        print(f"\nEpoch {epoch:03d} — Train Loss: {train_loss:.6f}")
        for name, m in val_metrics.items():
            print(f"  {name:8s} — MSE: {m['mse']:.6f}, MAE: {m['mae']:.6f}, R²: {m['r2']:.4f}")
    
    if avg_r2 > best_val_r2:
        best_val_r2 = avg_r2
        best_model_state = model.state_dict()
        counter = 0
    else:
        counter += 1
    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        model.load_state_dict(best_model_state)
        break
    
    scheduler.step(train_loss)

# Final evaluation & one-batch prediction
print("\nFinal Test Metrics:")
final_metrics = model.compute_metrics(test_loader)
for name, m in final_metrics.items():
    print(f"  {name:8s} — MSE: {m['mse']:.6f}, MAE: {m['mae']:.6f}, R²: {m['r2']:.4f}")

with torch.no_grad():
    xb, yb = next(iter(test_loader))
    xb, yb = xb.to(device), yb.to(device)
    out = model(xb)
    M_complex, sum_rate = model.reconstruct_M(out, xb.size(0))
    M_real_np = out[:, :12800].cpu().numpy().reshape(-1, 12800)
    M_imag_np = out[:, 12800:25600].cpu().numpy().reshape(-1, 12800)
    sum_rate_np = sum_rate.cpu().numpy().reshape(-1, 1)
    M_real_original = m_real_scaler.inverse_transform(M_real_np).reshape(xb.size(0), 20, 64, 10)
    M_imag_original = m_imag_scaler.inverse_transform(M_imag_np).reshape(xb.size(0), 20, 64, 10)
    sum_rate_original = rate_scaler.inverse_transform(sum_rate_np).flatten()
    M_complex_original = torch.complex(torch.tensor(M_real_original, device=device),
                                       torch.tensor(M_imag_original, device=device))
    # Compute mean absolute value of M
    M_abs_mean = torch.abs(M_complex_original).mean().item()
    print(f"\nSample Predictions — sum_rate (original scale): {sum_rate_original[:5]}")
    print(f"Reconstructed M_complex shape: {M_complex_original.shape}, dtype: {M_complex_original.dtype}")
    print(f"Mean absolute value of M_complex (original scale): {M_abs_mean:.6f}")

# Save the model and scalers
joblib.dump(model, "best_fcnn_model_adam_0.001.pkl")
joblib.dump(scaler, "scalar.pkl")
joblib.dump(rate_scaler, "rate_scaler.pkl")
joblib.dump(m_real_scaler, "m_real_scaler.pkl")
joblib.dump(m_imag_scaler, "m_imag_scaler.pkl")
print("Saved model and scalers: best_fcnn_model_adam_0.001.pkl, scalar.pkl, rate_scaler.pkl, m_real_scaler.pkl, m_imag_scaler.pkl")

Using device: cpu


C:\Users\Glason.WIN-06TKRSC40GC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training...

Epoch 000 — Train Loss: 0.742071
  M_real   — MSE: 0.182555, MAE: 0.258592, R²: 0.6058
  M_imag   — MSE: 0.182301, MAE: 0.259556, R²: 0.6063
  sum_rate — MSE: 0.248529, MAE: 0.394456, R²: 0.7349

Epoch 002 — Train Loss: 0.667381
  M_real   — MSE: 0.190901, MAE: 0.268075, R²: 0.5878
  M_imag   — MSE: 0.190631, MAE: 0.269005, R²: 0.5884
  sum_rate — MSE: 0.212715, MAE: 0.357544, R²: 0.7702

Epoch 004 — Train Loss: 0.655878
  M_real   — MSE: 0.189966, MAE: 0.267705, R²: 0.5898
  M_imag   — MSE: 0.189648, MAE: 0.268374, R²: 0.5904
  sum_rate — MSE: 0.216148, MAE: 0.371147, R²: 0.7673

Epoch 006 — Train Loss: 0.654939
  M_real   — MSE: 0.187368, MAE: 0.263714, R²: 0.5953
  M_imag   — MSE: 0.187190, MAE: 0.264152, R²: 0.5956
  sum_rate — MSE: 0.195581, MAE: 0.343847, R²: 0.7883

Epoch 008 — Train Loss: 0.651144
  M_real   — MSE: 0.186247, MAE: 0.266041, R²: 0.5978
  M_imag   — MSE: 0.185939, MAE: 0.266929, R²: 0.5984
  sum_rate — MSE: 0.193534, MAE: 0.334853, R²: 0.7897

Epoch 0

In [11]:


import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import math
import time

class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super(ResidualBlock, self).__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.bn = nn.BatchNorm1d(out_dim)
        self.relu = nn.LeakyReLU(0.12)
        self.dropout = nn.Dropout(0.15)
        self.residual = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)
        out = self.linear(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)
        return out + residual

class FCNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[1024, 512, 256, 128], output_dim_m=25600, output_dim_s=1):
        super(FCNN, self).__init__()
        # Shared backbone with residual connections
        backbone = []
        prev_dim = input_dim
        backbone.append(nn.Linear(prev_dim, hidden_dims[0]))
        backbone.append(nn.BatchNorm1d(hidden_dims[0]))
        backbone.append(nn.ReLU())
        backbone.append(nn.Dropout(0.2))
        prev_dim = hidden_dims[0]
        for dim in hidden_dims[1:]:
            backbone.append(ResidualBlock(prev_dim, dim))
            prev_dim = dim
        self.backbone = nn.Sequential(*backbone)
        
        # Simplified M branch (remove convolutional layers)
        self.m_head = nn.Sequential(
            nn.Linear(prev_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(512, output_dim_m),  # Output: 25600 (12800 real + 12800 imag)
            nn.Tanh()  # Constrain output to [-1, 1] for stable scaling
        )
        
        # Sum_rate branch
        self.s_head = nn.Sequential(
            nn.Linear(prev_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, output_dim_s)
        )

    def forward(self, x):
        features = self.backbone(x)
        m_out = self.m_head(features)  # Shape: [batch_size, 25600]
        s_out = self.s_head(features)  # Shape: [batch_size, 1]
        return torch.cat([m_out, s_out], dim=1)  # Shape: [batch_size, 25601]

    def reconstruct_M(self, output, batch_size):
        M_real = output[:, :12800].reshape(batch_size, 20, 64, 10)
        M_imag = output[:, 12800:25600].reshape(batch_size, 20, 64, 10)
        M_complex = torch.complex(M_real, M_imag)
        sum_rate = output[:, 25600]
        return M_complex, sum_rate

    def compute_metrics(self, loader):
        self.eval()
        totals = {k: 0 for k in ['mse_r','mae_r','r2_r','mse_i','mae_i','r2_i','mse_s','mae_s','r2_s']}
        n_samples = len(loader.dataset)
        with torch.no_grad():
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                out = self(xb)
                r_p, i_p, s_p = out[:, :12800], out[:, 12800:25600], out[:, 25600]
                r_t, i_t, s_t = yb[:, :12800], yb[:, 12800:25600], yb[:, 25600]
                for pred, true, tag, norm in [
                    (r_p, r_t, 'r', 12800),
                    (i_p, i_t, 'i', 12800),
                    (s_p, s_t, 's', 1)]:
                    totals[f'mse_{tag}'] += F.mse_loss(pred, true, reduction='sum').item()
                    totals[f'mae_{tag}'] += F.l1_loss(pred, true, reduction='sum').item()
                    ss_tot = torch.sum((true - true.mean())**2)
                    ss_res = torch.sum((true - pred)**2)
                    totals[f'r2_{tag}'] += (1 - (ss_res/(ss_tot+1e-9))).item()

        return {
            'M_real': {
                'mse': totals['mse_r']/(n_samples*12800),
                'mae': totals['mae_r']/(n_samples*12800),
                'r2':  totals['r2_r']/len(loader)
            },
            'M_imag': {
                'mse': totals['mse_i']/(n_samples*12800),
                'mae': totals['mae_i']/(n_samples*12800),
                'r2':  totals['r2_i']/len(loader)
            },
            'sum_rate': {
                'mse': totals['mse_s']/n_samples,
                'mae': totals['mae_s']/n_samples,
                'r2':  totals['r2_s']/len(loader)
            }
        }
    
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---- Load DL Model and Scalers ----
model_filename = "best_fcnn_model_adam_0.001.pkl"
scaler_filename = "scalar.pkl"
rate_scaler_filename = "rate_scaler.pkl"
m_real_scaler_filename = "m_real_scaler.pkl"
m_imag_scaler_filename = "m_imag_scaler.pkl"
try:
    fcnn_model = joblib.load(model_filename)
    scaler = joblib.load(scaler_filename)
    rate_scaler = joblib.load(rate_scaler_filename)
    m_real_scaler = joblib.load(m_real_scaler_filename)
    m_imag_scaler = joblib.load(m_imag_scaler_filename)
    print(f"Loaded model: {model_filename}")
    print(f"Loaded scaler: {scaler_filename}")
    print(f"Loaded rate scaler: {rate_scaler_filename}")
    print(f"Loaded M real scaler: {m_real_scaler_filename}")
    print(f"Loaded M imag scaler: {m_imag_scaler_filename}")
    fcnn_model.eval()
    fcnn_model.to(device)
except FileNotFoundError:
    print("Error: Model or scalers not found. Please ensure the files exist.")
    exit(1)

# Fixed MIMO configuration
Nt = 64
Nr = 64

# ---- Rayleigh Fading Channel Model ----
def RayleighChannel(Nr, Nt, L, train_size):
    H = (torch.randn(train_size, Nr, Nt, L, dtype=torch.cfloat, device=device) +
         1j * torch.randn(train_size, Nr, Nt, L, dtype=torch.cfloat, device=device)) / np.sqrt(2)
    print("Rayleigh Shape:", H.shape)
    return H

# ---- MMSE Beamforming ----
def MMSE_Beamforming(H, noise_var):
    batch_size, Nr_in, Nt_in = H.shape[:3]
    H_H = H.conj().transpose(1, 2)
    A = torch.bmm(H_H, H) + noise_var * torch.eye(Nt, dtype=torch.cfloat, device=device).unsqueeze(0)
    W = torch.linalg.solve(A, H_H)
    print("W's shape:", W.shape)
    return W

# ---- Excitation Matrix ----
def genE(lamd, de, Nt, RFno):
    x = torch.arange(Nt, device=device) * de
    y = Nt * de / 2 * torch.ones(Nt, device=device)
    xf = torch.arange(RFno, device=device) * de + de / 2
    yf = Nt * de / 2 * torch.ones(RFno, device=device)
    
    dist = torch.sqrt((x.unsqueeze(1) - xf.unsqueeze(0))**2 + 
                      (y.unsqueeze(1) - yf.unsqueeze(0))**2)
    E = torch.exp(-1j * 2 * np.pi * math.sqrt(3) / lamd * dist)
    print("E's shape:", E.shape)
    return E.unsqueeze(0)

# ---- Gradient Projection Algorithm (GP) ----
def sum_capacity(w, H, M, batch_size):
    Nr_in, L = w.shape[1], w.shape[2]
    obj = torch.zeros(batch_size, dtype=torch.float, device=device)
    
    for l in range(L):
        wt = w[:, :, l]
        H_l = H[:, :, :, l]
        signal = torch.bmm(wt.unsqueeze(1).conj(), torch.bmm(H_l, M))
        power = torch.abs(signal.squeeze(1))**2 / (Nr_in + 1e-9)
        obj += torch.log2(1 + power.sum(dim=1))
    
    obj_np = obj.cpu().numpy().reshape(-1, 1)
    obj_normalized = rate_scaler.transform(obj_np).flatten()
    return torch.tensor(obj_normalized, dtype=torch.float, device=device)

def GP(H, E, L, hypm, Pt):
    Tno = H.shape[0]
    RFno = E.shape[2]
    
    w = torch.randn(Tno, Nr, L, dtype=torch.cfloat, device=device)
    w = w / (torch.abs(w) + 1e-9)
    m = torch.rand(Tno, Nt, RFno, dtype=torch.float, device=device)
    M = m * E
    print("Initial M shape:", M.shape)
    
    obj0 = torch.tensor(0.0, device=device)
    max_iter = 100
    iter_count = 0
    
    while iter_count < max_iter:
        gradm = torch.zeros(Tno, Nt, RFno, dtype=torch.float, device=device)
        
        for l in range(L):
            wt = w[:, :, l]
            H_l = H[:, :, :, l]
            signal = torch.bmm(wt.unsqueeze(1).conj(), torch.bmm(H_l, M))
            denominator = (torch.abs(signal)**2 + Nr).squeeze(1)
            H_permuted = H_l.permute(0, 2, 1)
            numerator = torch.bmm(H_permuted, wt.unsqueeze(2).conj())
            gradm += (numerator * E / denominator.unsqueeze(1)).real
        
        m = torch.clamp(m + hypm * gradm, 0, 1)
        M = m * E
        
        w_new = torch.zeros(Tno, Nr, L, dtype=torch.cfloat, device=device)
        for l in range(L):
            H_l = H[:, :, :, l]
            w_new[:, :, l] = torch.bmm(H_l, M).mean(dim=2)
        w = w_new / (torch.abs(w_new) + 1e-9)
        
        obj1 = sum_capacity(w, H, M, Tno)
        print("obj1 (normalized sum capacity):", obj1.mean().item())
        if torch.abs(obj1.mean() - obj0) / (obj1.mean() + 1e-9) <= 1e-4:
            print("Convergence reached. Stopping optimization.")
            break
        obj0 = obj1.mean()
        iter_count += 1
    
    return obj1, w, M

# ---- Simulation Parameters ----
dmode = 1
L = 3
lamd = 0.04
scale = 1e9
SNR = 23.9
Pt = 10 ** (SNR / 10) * (1 / scale) ** 2
deH = lamd / 5
test_size = 20
stepSize1 = 0.5e1
noise_var = 0.001
Nt = 64
Nr = 64

if dmode == 1:
    lopPara = [0, 10, 20, 30, 40]
elif dmode == 2:
    lopPara = [2, 4, 6, 8, 10]

# Storage for results
RateAll_theory = torch.zeros([len(lopPara)], device=device)
TimeAll_theory = np.zeros((len(lopPara)))
RateAll_dl = torch.zeros([len(lopPara)], device=device)
TimeAll_dl = np.zeros((len(lopPara)))
M_means_theory = []
M_means_dl = []

# ---- Generate Input Features for DL Model ----
def generate_features(RFno, Nx):
    features = np.array([[L, lamd, scale, SNR, Pt, deH, test_size, RFno, stepSize1, noise_var, Nx]])
    return features

# ---- Modified reconstruct_M to Handle Variable RFno ----
def reconstruct_M(model, output, batch_size, RFno):
    max_rf = 10  # Model outputs fixed 10 RF chains
    elements_per_rf = 20 * 64  # Elements per RF chain
    M_real = output[:, :12800].reshape(batch_size, 20, 64, max_rf)
    M_imag = output[:, 12800:25600].reshape(batch_size, 20, 64, max_rf)
    # Truncate to actual RFno
    M_real = M_real[:, :, :, :RFno]
    M_imag = M_imag[:, :, :, :RFno]
    M_complex = torch.complex(M_real, M_imag)
    sum_rate = output[:, 25600]
    return M_complex, sum_rate

# ---- Function to Format Matrix Display ----
def format_matrix(M, name, max_elements=5, precision=4):
    M_np = M.cpu().numpy()
    print(f"{name} (shape: {M.shape}):")
    # Display first few elements
    flat = M_np.flatten()[:max_elements]
    formatted = np.array2string(flat, precision=precision, suppress_small=True)
    print(f"First {min(max_elements, len(flat))} elements: {formatted}")
    # Summary statistics
    abs_M = np.abs(M_np)
    print(f"Mean: {abs_M.mean():.6f}, Std: {abs_M.std():.6f}, "
          f"Min: {abs_M.min():.6f}, Max: {abs_M.max():.6f}")

# ---- Main Simulation Loop ----
np.set_printoptions(precision=4, suppress=True)  # Clean NumPy output
with torch.no_grad():
    for lop in range(len(lopPara)):
        print(f"\nIteration {lop + 1}/{len(lopPara)}: Nx = {lopPara[lop]}")
        Nx = int(lopPara[lop])

        RFno = min(max(1, Nx // 5), 9)

        # --- Theoretical Simulation ---
        H_RHS = RayleighChannel(Nr, Nt, L, test_size) * scale
        H_BS = H_RHS.mean(dim=3)
        W_MMSE = MMSE_Beamforming(H_BS, noise_var)
        
        H_testD = torch.zeros(test_size, Nr, Nt, L, dtype=torch.cfloat, device=device)
        for l in range(L):
            H_testD[:, :, :, l] = torch.bmm(H_RHS[:, :, :, l], W_MMSE)
        
        E = genE(lamd, deH, Nt, RFno).repeat(test_size, 1, 1)

        print("H_testD shape:", H_testD.shape)
        print("E shape:", E.shape)
        
        time_start_theory = time.time()
        gp_rates, w, M_theory = GP(H_testD, E, L, stepSize1, Pt)
        time_end_theory = time.time()

        M_mean_theory = torch.abs(M_theory).mean().item()
        M_means_theory.append(M_mean_theory)

        Rate_theory_normalized = gp_rates.mean().item()
        RateAll_theory[lop] = Rate_theory_normalized
        TimeAll_theory[lop] = time_end_theory - time_start_theory

        # --- DL Model Prediction ---
        Rate_pred = float('nan')
        M_mean_pred = float('nan')
        M_pred = None
        time_dl = 0.0
        try:
            X = generate_features(RFno, Nx)
            X_scaled = scaler.transform(X)
            X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device)

            time_start_dl = time.time()
            pred = fcnn_model(X_tensor)
            time_end_dl = time.time()
            M_pred, Rate_pred_normalized = reconstruct_M(fcnn_model, pred, batch_size=1, RFno=RFno)
            
            # Inverse-transform outputs
            M_real_np = pred[:, :RFno*20*64].cpu().numpy().reshape(-1, RFno*20*64)
            M_imag_np = pred[:, 12800:12800+RFno*20*64].cpu().numpy().reshape(-1, RFno*20*64)
            Rate_pred_np = Rate_pred_normalized.cpu().numpy().reshape(-1, 1)
            M_real_original = m_real_scaler.inverse_transform(M_real_np).reshape(1, 20, 64, RFno)
            M_imag_original = m_imag_scaler.inverse_transform(M_imag_np).reshape(1, 20, 64, RFno)
            Rate_pred = rate_scaler.inverse_transform(Rate_pred_np).flatten()[0]
            M_pred_original = torch.complex(torch.tensor(M_real_original, device=device),
                                            torch.tensor(M_imag_original, device=device))

            # Post-process M to match theoretical magnitude (~0.5)
            M_mean_pred = torch.abs(M_pred_original).mean().item()
            if M_mean_pred < 0.1 or M_mean_pred > 1.0:  # Check for scaling issues
                scaling_factor = 0.5 / (M_mean_pred + 1e-9)
                M_pred_original *= scaling_factor
                M_mean_pred = torch.abs(M_pred_original).mean().item()
                print(f"Applied scaling factor {scaling_factor:.4f} to M_pred_original")

            RateAll_dl[lop] = Rate_pred_normalized.item()
            TimeAll_dl[lop] = time_end_dl - time_start_dl
            M_means_dl.append(M_mean_pred)
        except Exception as e:
            print(f"DL prediction failed: {e}")
            Rate_pred = float('nan')
            M_mean_pred = float('nan')
            M_means_dl.append(float('nan'))

        Rate_theory = rate_scaler.inverse_transform([[Rate_theory_normalized]])[0, 0]

        # Print Results with Improved Display
        print(f"RFno: {RFno}")
        format_matrix(M_theory[0], "Theoretical M", max_elements=5)
        if M_pred is not None:
            format_matrix(M_pred_original[0], "Predicted M", max_elements=5)
        else:
            print("Predicted M: Not available due to prediction failure")
        print(f"Rate (Theoretical, original scale): {Rate_theory:.6f}")
        print(f"Rate (DL, original scale): {Rate_pred:.6f}")
        print(f"Computation Time (Theoretical): {TimeAll_theory[lop]:.6f} seconds")
        print(f"Computation Time (DL): {TimeAll_dl[lop]:.6f} seconds")
        print(f"Sum Rate (Theoretical, normalized for storage): {Rate_theory_normalized:.6f}")

        # Plot Magnitude Distribution
        plt.figure(figsize=(8, 6))
        plt.hist(torch.abs(M_theory[0]).cpu().numpy().flatten(), bins=50, alpha=0.5, label='Theoretical M', density=True)
        if M_pred is not None:
            plt.hist(torch.abs(M_pred_original[0]).cpu().numpy().flatten(), bins=50, alpha=0.5, label='Predicted M', density=True)
        plt.xlabel('Magnitude')
        plt.ylabel('Density')
        plt.title(f'Magnitude Distribution (Nx={Nx}, RFno={RFno})')
        plt.legend()
        plt.grid()
        plt.savefig(f'magnitude_distribution_Nx_{Nx}.png')
        plt.close()

# ---- Plot Results ----
RateAll_theory_np = RateAll_theory.cpu().numpy().reshape(-1, 1)
RateAll_dl_np = RateAll_dl.cpu().numpy().reshape(-1, 1)
RateAll_theory_original = rate_scaler.inverse_transform(RateAll_theory_np).flatten()
RateAll_dl_original = rate_scaler.inverse_transform(RateAll_dl_np).flatten()

# Sum Rate Plot
plt.figure(figsize=(8, 6))
plt.plot(lopPara, RateAll_theory_original, 'b-*', label='RHS (Gradient Projection (GP))')
plt.plot(lopPara, RateAll_dl_original, 'r-*', label='RHS (DL Predicted)')
plt.xlabel('Nx: Number of RHS Elements')
plt.ylabel('Sum Rate (bps/Hz)')
plt.title('RHS Elements vs. Sum Rate')
plt.legend()
plt.grid()
plt.savefig('sum_rate_plot.png')
plt.close()

# Computation Time Plot
plt.figure(figsize=(8, 6))
plt.semilogy(lopPara, TimeAll_theory / test_size, 'b-*', label='RHS (Gradient Projection (GP))')
plt.semilogy(lopPara, TimeAll_dl / test_size, 'r-*', label='RHS (DL Inference)')
plt.xlabel('Nx: Number of RHS Elements')
plt.ylabel('Computation Time per Sample (s)')
plt.title('RHS Elements vs. Computation Time')
plt.legend()
plt.grid()
plt.savefig('computation_time_plot.png')
plt.close()

# M Mean Absolute Value Plot
plt.figure(figsize=(8, 6))
plt.plot(lopPara, M_means_theory, 'b-*', label='Theoretical M Mean Abs')
plt.plot(lopPara, M_means_dl, 'r-*', label='Predicted M Mean Abs')
plt.xlabel('Nx: Number of RHS Elements')
plt.ylabel('Mean Absolute Value of M')
plt.title('RHS Elements vs. M Mean Absolute Value')
plt.legend()
plt.grid()
plt.savefig('m_mean_abs_plot.png')
plt.close()

Using device: cpu
Loaded model: best_fcnn_model_adam_0.001.pkl
Loaded scaler: scalar.pkl
Loaded rate scaler: rate_scaler.pkl
Loaded M real scaler: m_real_scaler.pkl
Loaded M imag scaler: m_imag_scaler.pkl

Iteration 1/5: Nx = 0
Rayleigh Shape: torch.Size([20, 64, 64, 3])
W's shape: torch.Size([20, 64, 64])
E's shape: torch.Size([64, 1])
H_testD shape: torch.Size([20, 64, 64, 3])
E shape: torch.Size([20, 64, 1])
Initial M shape: torch.Size([20, 64, 1])
obj1 (normalized sum capacity): -1.2587279081344604
Convergence reached. Stopping optimization.
Applied scaling factor 9.5065 to M_pred_original
RFno: 1
Theoretical M (shape: torch.Size([64, 1])):
First 5 elements: [ 0.3769-0.7196j  0.4581-0.8745j -0.2324+0.0288j  0.0565+0.0633j
  0.2131-0.8862j]
Mean: 0.460667, Std: 0.306728, Min: 0.000000, Max: 1.000000
Predicted M (shape: torch.Size([20, 64, 1])):
First 5 elements: [ 1.5807-2.6575j -1.2724+0.1521j -0.0346-0.0486j -0.1425+0.0885j
  0.1966-0.0742j]
Mean: 0.500000, Std: 0.896118, Min: 0.0